In [1]:
import pandas as pd

Ir = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/IrLumDB.csv')
Ir

,L1,L2,L3,Counterion,Abbreviation_in_the_article,Charge,Max_wavelength(nm),PLQY,tau(s*10^-6),Solvent,DOI,Notes,PLQY_in_train
0,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C([O-])c1ccccn1,NaN,1,0,553,0.2500,4.32,CH2Cl2,10.3390/molecules27010232,NaN,1
1,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,c1ccc(-c2ccccn2)nc1,FP(F)(F)(F)(F)F,2,1,570,0.1700,6.14,CH2Cl2,10.3390/molecules27010232,NaN,1
2,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1,FP(F)(F)(F)(F)F,3,1,595,0.1400,0.82,CH3OH,10.3390/molecules27010232,NaN,1
3,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=S(=O)([O-])c1ccc(P(c2ccc(S(=O)(=O)O[Na])cc2)...,NaN,4,0,555,0.0435,41.87,PBS,10.3390/molecules27010232,NaN,1
4,[c-]1ccccc1-c1ccccn1,[c-]1ccccc1-c1ccccn1,CCOP(=O)(OCC)c1ccnc(-c2cc(P(=O)(OCC)OCC)ccn2)c1,FP(F)(F)(F)(F)F,Ir(ppy)2bP,1,667,0.0020,NaN,CH3OH,10.1002/adhm.202100706,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,O=C([O-])c1ccccn1,NaN,(m-CzFSOPy)2IrPic,0,554,0.3810,0.64,CH2Cl2,10.1016/j.dyepig.2018.07.019,NaN,1
1450,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir1-me,0,608,0.4000,2.12,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1
1451,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir2-cf3,0,604,0.4100,2.21,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1
1452,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PQZ-Ir3-me,0,628,0.6000,2.06,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1


In [4]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import traceback

# ============================ КОНФИГУРАЦИЯ ============================
# Убедись, что твой DataFrame называется df и содержит колонки L1, L2, L3
# df = pd.read_csv("your_ligands.csv")   # <-- раскомментируй, если нужно загрузить

# =====================================================================

def find_coordinating_atoms(mol):
    """Возвращает список индексов атомов, которые могут координироваться к Ir"""
    coord_atoms = []
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        charge = atom.GetFormalCharge()
        idx = atom.GetIdx()

        # Анионный углерод C-
        if symbol == 'C' and charge == -1:
            coord_atoms.append(idx)
        # Азот в ароматическом кольце (пиридин, имидазол и т.д.)
        elif symbol == 'N' and atom.GetIsAromatic():
            coord_atoms.append(idx)
        # Кислород в карбоксилате O-
        elif symbol == 'O' and charge == -1:
            coord_atoms.append(idx)
    return coord_atoms


def create_iridium_complex_from_smiles(l1, l2, l3, row_index=None):
    """Создаёт [Ir(L1)(L2)(L3)] комплекс из трёх SMILES-строк"""
    ligand_smiles = [l1, l2, l3]
    ligand_names = ["L1", "L2", "L3"]

    # Проверяем, что все SMILES — строки, а не NaN
    if any(pd.isna(s) or not isinstance(s, str) for s in ligand_smiles):
        return None, "Один из лигандов пустой или не строка"

    rwmol = Chem.RWMol()
    ir = Chem.Atom(77)  # Иридий
    ir_idx = rwmol.AddAtom(ir)

    all_coord_atoms = []

    for i, smiles in enumerate(ligand_smiles):
        name = ligand_names[i]
        mol = Chem.MolFromSmiles(smiles.strip())
        if mol is None:
            return None, f"Некорректный SMILES в {name}: {smiles}"

        # Смещение индексов
        offset = rwmol.GetNumAtoms()

        # Копируем атомы
        atom_map = {}
        for atom in mol.GetAtoms():
            new_atom = Chem.Atom(atom.GetSymbol())
            new_atom.SetFormalCharge(atom.GetFormalCharge())
            new_atom.SetIsAromatic(atom.GetIsAromatic())
            new_idx = rwmol.AddAtom(new_atom)
            atom_map[atom.GetIdx()] = new_idx

        # Копируем связи
        for bond in mol.GetBonds():
            a1 = atom_map[bond.GetBeginAtomIdx()]
            a2 = atom_map[bond.GetEndAtomIdx()]
            rwmol.AddBond(a1, a2, bond.GetBondType())

        # Находим координационные атомы в текущем лиганде
        coord_local = find_coordinating_atoms(mol)
        if not coord_local:
            return None, f"Не найдены координационные атомы в {name}"

        coord_shifted = [idx + offset for idx in coord_local]
        all_coord_atoms.extend(coord_shifted)

        # Создаём дативные связи Ir←N/C/O
        for c in coord_shifted:
            rwmol.AddBond(ir_idx, c, Chem.BondType.DATIVE)

    # Устанавливаем заряд иридия (чтобы комплекс был нейтральным)
    total_ligand_charge = sum(
        a.GetFormalCharge() for a in rwmol.GetAtoms()
        if a.GetAtomicNum() != 77
    )
    rwmol.GetAtomWithIdx(ir_idx).SetFormalCharge(-total_ligand_charge)

    # Финализируем молекулу
    mol = rwmol.GetMol()
    try:
        Chem.SanitizeMol(mol)
    except Exception as e:
        print(f"   [Строка {row_index}] Санитизация не удалась: {e}")

    try:
        complex_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return complex_smiles, "OK"
    except Exception as e:
        return None, f"Ошибка генерации SMILES: {e}"


# ============================ ОСНОВНОЙ ЦИКЛ ============================

def process_dataframe(df):
    # Создаём новые колонки
    df['Complex_SMILES'] = None
    df['Complex_Status'] = None

    print(f"Обрабатываем {len(df)} строк...")

    for idx in df.index:
        print(f"\nОбрабатываем строку {idx}...")

        try:
            l1 = df.at[idx, 'L1']
            l2 = df.at[idx, 'L2']
            l3 = df.at[idx, 'L3']

            smiles, status = create_iridium_complex_from_smiles(l1, l2, l3, row_index=idx)

            df.at[idx, 'Complex_SMILES'] = smiles
            df.at[idx, 'Complex_Status'] = status

            if status == "OK":
                print(f"   Успех! {smiles[:80]}...")
            else:
                print(f"   Ошибка: {status}")

        except Exception as e:
            print(f"   Критическая ошибка в строке {idx}: {e}")
            traceback.print_exc()
            df.at[idx, 'Complex_Status'] = f"Exception: {str(e)}"

    return df


# ============================ ЗАПУСК ============================

# Если df ещё не в памяти — загрузи его:
# df = pd.read_csv("your_file.csv")

# Запускаем обработку
df = process_dataframe(Ir)

# Сохраняем результат
df.to_csv("iridium_complexes_generated.csv", index=False)
df.to_pickle("iridium_complexes_generated.pkl")  # удобнее для дальнейшей работы

print("\nГОТОВО! Результат сохранён в:")
print("   iridium_complexes_generated.csv")
print("   iridium_complexes_generated.pkl")

# Показываем первые успешные комплексы
successful = df[df['Complex_Status'] == 'OK']
print(f"\nУспешно создано комплексов: {len(successful)} из {len(df)}")
if len(successful) > 0:
    print("\nПримеры SMILES комплексов:")
    print(successful['Complex_SMILES'].head(10).to_list())

Обрабатываем 1454 строк...

Обрабатываем строку 0...
   [Строка 0] Санитизация не удалась: Explicit valence for atom # 20 N, 5, is greater than permitted
   Успех! O=C1[O-]<-[Ir+3]234567(->[c-]8cc(-c9[n]<-2c2c%10ccccc%10c%10ccccc%10c2[n]<-39-c2...

Обрабатываем строку 1...
   [Строка 1] Санитизация не удалась: Explicit valence for atom # 20 N, 5, is greater than permitted
   Успех! c1ccc(-[n]23<-[Ir+2]456789(->[c-]%10cc(-c2[n]<-4c2c4ccccc4c4ccccc4c23)ccc%10-c2c...

Обрабатываем строку 2...
   [Строка 2] Санитизация не удалась: Explicit valence for atom # 20 N, 5, is greater than permitted
   Успех! O=C(O)c1cc[n]2<-[Ir+2]345678(->[c-]9cc(-c%10[n]<-3c3c%11ccccc%11c%11ccccc%11c3[n...

Обрабатываем строку 3...
   [Строка 3] Санитизация не удалась: Explicit valence for atom # 20 N, 5, is greater than permitted
   Успех! O=S(=O)([O][Na])c1ccc(P(c2ccc(S(=O)(=O)[O][Na])cc2)c2ccccc2P(c2ccc(S(=O)(=O)[O][...

Обрабатываем строку 4...
   [Строка 4] Санитизация не удалась: Can't kekulize mol.  Unke

[20:58:59] Explicit valence for atom # 20 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 20 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 20 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 20 N, 5, is greater than permitted
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 3 4 6 13 14
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 8 9 11 18 19
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 9 10 12 19 20
[20:58:59] Exp

   Успех! O=C1[O-]<-[Ir+3]2345(->[c-]6ccccc6-c6c(-c7ccccc7)[n]<-2c2cc(F)c(F)cc2[n]<-36)(->...

Обрабатываем строку 160...
   [Строка 160] Санитизация не удалась: Explicit valence for atom # 41 O, 2, is greater than permitted
   Успех! Cc1cc2c(cc1C)[n]1<-[Ir+3]34567(->[O-]C(=O)c8c[n]<-3cc[n]<-48)(->[c-]3ccccc3-c1c(...

Обрабатываем строку 161...
   [Строка 161] Санитизация не удалась: Explicit valence for atom # 51 O, 2, is greater than permitted
   Успех! Cc1cc2c(cc1C)[n]1<-[Ir+3]34567(->[O-]C(=O)c8c[n]<-3cc[n]<-48)(->[c-]3ccccc3-c1c(...

Обрабатываем строку 162...
   [Строка 162] Санитизация не удалась: Explicit valence for atom # 51 O, 2, is greater than permitted
   Успех! O=C1[O-]<-[Ir+3]23456(->[c-]7ccccc7-c7c(-c8ccccc8)[n]<-2c2cc(F)c(F)cc2[n]<-37)(-...

Обрабатываем строку 163...
   [Строка 163] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
   Успех! CCOP(=O)(Cc1cc[n]2<-[Ir+2]34(->[c-]5ccccc5-c5cccc[n]<-35)(->[c-]3ccccc3-c3cccc[n...

Обрабатываем строк

[20:58:59] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 5 7 14 15 20
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 5 7 14 15 20
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 5 7 14 15 20
[20:58:59] Explicit valence for atom # 29 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 35 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 45 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 23 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 23 N, 5, is greater than permitted
[20:58:59] Explicit valence for atom # 3 O, 2, is greater than permitted
[20:58:59] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 7 9 10 11 12 13 14 16 17 18 19 20 21 22
[20:58:59] Explicit valence for atom # 47 O, 2, is greater than permitted
[20:58:59] Explicit valence for atom # 49 O, 2, is greater than permitted
[20:58:59] Explicit valence for atom # 47 O, 2, is grea

   Успех! N#Cc1cc[n]2<-[Ir+2]34(->[c-]5ccccc5-c5cccc[n]<-35)(->[c-]3ccccc3-c3cccc[n]<-43)-...

Обрабатываем строку 297...
   [Строка 297] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 5 12 14
   Успех! N#Cc1cc[n]2<-[Ir+2]34(->[c-]5cc(F)cc(F)c5-c5cccc[n]<-35)(->[c-]3cc(F)cc(F)c3-c3c...

Обрабатываем строку 298...
   [Строка 298] Санитизация не удалась: Explicit valence for atom # 51 O, 2, is greater than permitted
   Успех! CC(=O)C=C(C)[O-]<-[Ir+3]12(->[c-]3ccccc3-c3ccc4c(-c5ccccc5)cccc4[n]<-13)->[c-]1c...

Обрабатываем строку 299...
   [Строка 299] Санитизация не удалась: Explicit valence for atom # 41 O, 2, is greater than permitted
   Успех! CC(=O)C=C(C)[O-]<-[Ir+3]12(->[c-]3ccccc3-c3ccc4c(F)cccc4[n]<-13)->[c-]1ccccc1-c1...

Обрабатываем строку 300...
   [Строка 300] Санитизация не удалась: Explicit valence for atom # 53 O, 2, is greater than permitted
   Успех! CC(=O)C=C(C)[O-]<-[Ir+3]12(->[c-]3ccccc3-c3ccc4c(-c5ccc(F)cc5)cccc4[n]<-13)->[c-...

Обрабатываем стр

...

[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 25 26 27 28 30 41 42
[20:59:01] Explicit valence for atom # 26 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 32 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 34 N, 5, is greater than permitted
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 23 24 25
[20:59:01] Explicit valence for atom # 53 O, 2, is greater than permitted
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 51 52 54
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 3 4 6 24 25
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 26 28
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 30 32
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 34 36
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 34 36
[20:59:01] Explicit valence for atom # 9 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 9 N, 5, is greater than permitted
[20:59:01]

   [Строка 1135] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 7 9 10 11 12 13 14
   Успех! c1ccc(-c2cc[n]3<-[Ir+2]45(->[c-]6ccccc6-c6sc7ccccc7[n]<-46)(->[c-]4ccccc4-c4sc6c...

Обрабатываем строку 1136...
   [Строка 1136] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 4 6 13
   Успех! Cc1cc[c-]2<-[Ir+2]34(->[c-]5ccc(C)cc5-c5cccc[n]<-35)(->[n]3ccccc3-c2c1)->[n]1c(C...

Обрабатываем строку 1137...
   [Строка 1137] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
   Успех! c1cc[c-]2<-[Ir+2]34(->[c-]5ccccc5-c5ccc6ccccc6[n]<-35)(->[n]3cccc5ccc6ccc[n]<-4c...

Обрабатываем строку 1138...
   [Строка 1138] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
   Успех! Cc1cc(-c2ccccc2)c2ccc3c(-c4ccccc4)cc(C)[n]4<-[Ir+2]56(->[c-]7ccccc7-c7ccc8ccccc8...

Обрабатываем строку 1139...
   [Строка 1139] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
   Успех! c1ccc(-c2cc[n]3<-[Ir+2]45(->[c-]6

[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 5
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 5
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 10 11 12 13 14 15
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 10 11 12 13 14 15
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 5
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 10 11 12 13 14 15
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Explicit valence for atom # 40 N, 4, is greater than permitted
[20:59:01] Explicit valence for atom # 36 N, 4, is greater than permitted
[20:59:01] Explicit valence for atom # 44 N, 4, is greater than permitted
[20:59:01] Explicit valence for atom # 44 N, 4, is greater than permitted
[20:59:01] Explicit valence for atom # 14 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 14 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 13 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom 

   [Строка 1287] Санитизация не удалась: Explicit valence for atom # 88 N, 4, is greater than permitted
   Успех! COc1c[c-]2<-[Ir+3]345(->[c-]6cc(OC)c(S(=O)(=O)c7ccc(C)cc7)c(F)c6-c6cc(-c7c(C)cc(...

Обрабатываем строку 1288...
   [Строка 1288] Санитизация не удалась: Explicit valence for atom # 78 N, 4, is greater than permitted
   Успех! COc1c[c-]2<-[Ir+3]345(->[c-]6cc(OC)c(S(C)(=O)=O)c(OC)c6-c6cc(-c7c(C)cc(C)cc7C)cc...

Обрабатываем строку 1289...
   [Строка 1289] Санитизация не удалась: Explicit valence for atom # 82 N, 4, is greater than permitted
   Успех! COc1c[c-]2<-[Ir+3]345(->[c-]6cc(OC)c(S(=O)(=O)c7ccc(C)cc7)c(F)c6-c6cc(-c7c(C)cc(...

Обрабатываем строку 1290...
   [Строка 1290] Санитизация не удалась: Explicit valence for atom # 72 N, 4, is greater than permitted
   Успех! COc1c[c-]2<-[Ir+3]345(->[c-]6cc(OC)c(S(C)(=O)=O)c(OC)c6-c6cc(-c7c(C)cc(C)cc7C)cc...

Обрабатываем строку 1291...
   [Строка 1291] Санитизация не удалась: Can't kekulize mol.  Unkekulized atoms: 3 4 6 17 18

[20:59:01] Explicit valence for atom # 44 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 61 N, 5, is greater than permitted
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 37 39 40
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 37 39 40 41 42 43 44
[20:59:01] Explicit valence for atom # 37 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 37 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 37 N, 5, is greater than permitted
[20:59:01] Explicit valence for atom # 37 N, 5, is greater than permitted
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[20:59:01] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5

In [16]:
import pandas as pd
from rdkit import Chem
import traceback

# ============================================================
# НАСТРОЙКИ
# ------------------------------------------------------------
# Ожидается, что у тебя уже есть DataFrame Ir с колонками 'L1', 'L2', 'L3'
# Например:
# Ir = pd.read_csv("your_ligands.csv")
#
# Код внизу:
# Ir = process_dataframe_canonical(Ir)
# Ir.to_csv("iridium_complexes_canonical.csv", index=False)
# ============================================================

HALOGENS = {"F", "Cl", "Br", "I"}
MAIN_DONORS = {"C", "N", "O", "S", "P"}  # то, что хотим "повернуть" к Ir


def find_primary_donor_atom(mol):
    """
    Возвращает индекс "главного" донорного атома лиганда по эвристике:
    приоритет (чем меньше score, тем важнее):
      0) C-  (циклометаллированный углерод)
      1) ароматический N
      2) O-
      3) S/P с зарядом <= 0
      10) галогены X- (F-, Cl-, Br-, I-), если больше ничего нет
    Если подходящих атомов нет, возвращает None.
    """
    candidates = []

    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        ch = atom.GetFormalCharge()
        idx = atom.GetIdx()
        arom = atom.GetIsAromatic()

        score = None

        if sym == "C" and ch == -1:
            score = 0
        elif sym == "N" and arom:
            score = 1
        elif sym == "O" and ch == -1:
            score = 2
        elif sym in {"S", "P"} and ch <= 0:
            score = 3
        elif sym in HALOGENS and ch == -1:
            score = 10

        if score is not None:
            candidates.append((score, idx))

    if not candidates:
        return None

    candidates.sort()
    return candidates[0][1]


def canonical_ligand_smiles_rooted_at_donor(smiles):
    """
    Берёт SMILES лиганда и возвращает кортеж:
        (lig_smiles, donor_symbol, total_charge)

    где lig_smiles — канонический для данного лиганда SMILES,
    "заякоренный" (rootedAtAtom) на донорном атоме.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Некорректный SMILES лиганда: {smiles}")

    donor_idx = find_primary_donor_atom(mol)
    if donor_idx is None:
        raise ValueError(f"Не найден донорный атом в лигандe: {smiles}")

    lig_smiles = Chem.MolToSmiles(
        mol,
        isomericSmiles=True,
        rootedAtAtom=donor_idx
        # canonical здесь по сути выключается при rootedAtAtom,
        # но порядок и запись для данного root детерминированы.
    )

    donor_symbol = mol.GetAtomWithIdx(donor_idx).GetSymbol()
    total_charge = sum(a.GetFormalCharge() for a in mol.GetAtoms())

    return lig_smiles, donor_symbol, total_charge


def ir_charge_from_ligand_charges(lig_charges):
    """
    Считает заряд Ir так, чтобы комплекс в целом был нейтрален.
    Если хочешь всегда [Ir+3], можешь просто вернуть 3.
    """
    total_ligand_charge = sum(lig_charges)
    return -total_ligand_charge


def ir_charge_to_bracket(charge):
    """
    Преобразует целый заряд Ir в строку вида:
      0   -> [Ir]
      +3  -> [Ir+3]
      -2  -> [Ir-2]
    """
    if charge == 0:
        return "[Ir]"
    if charge > 0:
        return f"[Ir+{charge}]"
    # отрицательный заряд
    return f"[Ir{charge}]"


def make_canonical_ir_complex_smiles(l1, l2, l3):
    """
    Делает канонический (по нашим правилам) SMILES комплекса [Ir(L1)(L2)(L3)]:

    1) Каждый лиганд записывается SMILES, начинающимся с донорного атома.
    2) Лиганды сортируются:
        - сперва доноры C/N/O/S/P (MAIN_DONORS),
        - потом галогены (HALOGENS),
        - затем вообще прочее (на всякий случай).
    3) Заряд Ir выбирается так, чтобы суммарный заряд комплекса был 0.

    Возвращает строку вида:
        [Ir+q](Lig1_SMILES)(Lig2_SMILES)(Lig3_SMILES)
    """
    ligands_raw = [l1, l2, l3]
    ligands_info = []   # (priority_group, donor_symbol, lig_smiles, lig_charge)
    ligand_charges = []

    for smi in ligands_raw:
        if pd.isna(smi) or not isinstance(smi, str):
            raise ValueError(f"Один из лигандов пустой или не строка: {smi}")

        lig_smiles, donor_symbol, charge = canonical_ligand_smiles_rooted_at_donor(smi)
        ligand_charges.append(charge)

        # приоритет: 0 — C/N/O/S/P, 1 — галогены, 2 — другое
        if donor_symbol in MAIN_DONORS:
            group = 0
        elif donor_symbol in HALOGENS:
            group = 1
        else:
            group = 2

        ligands_info.append((group, donor_symbol, lig_smiles, charge))

    # сортируем по (group, lig_smiles) — это задаёт детерминированный порядок
    ligands_info_sorted = sorted(ligands_info, key=lambda x: (x[0], x[2]))
    ligand_smiles_sorted = [x[2] for x in ligands_info_sorted]

    # считаем заряд Ir (или поставь константу +3, если нужно строго [Ir+3])
    ir_q = ir_charge_from_ligand_charges(ligand_charges)
    ir_block = ir_charge_to_bracket(ir_q)

    complex_smiles = ir_block + "(" + ")(".join(ligand_smiles_sorted) + ")"
    return complex_smiles


def create_iridium_complex_from_ligands_canonical(l1, l2, l3, row_index=None):
    """
    Обёртка для одного ряда DataFrame:
    возвращает (complex_smiles, status_string)
    """
    try:
        complex_smiles = make_canonical_ir_complex_smiles(l1, l2, l3)
        return complex_smiles, "OK"
    except Exception as e:
        msg = f"Ошибка генерации SMILES (строка {row_index}): {e}"
        return None, msg


def process_dataframe_canonical(df):
    """
    Обрабатывает DataFrame с колонками 'L1', 'L2', 'L3' и
    добавляет:
      - 'Complex_SMILES_canonical'
      - 'Complex_Status_canonical'
    """
    df = df.copy()
    df['Complex_SMILES_canonical'] = None
    df['Complex_Status_canonical'] = None

    print(f"Обрабатываем {len(df)} строк для канонических Ir-комплексов...")

    for idx in df.index:
        try:
            l1 = df.at[idx, 'L1']
            l2 = df.at[idx, 'L2']
            l3 = df.at[idx, 'L3']

            smiles, status = create_iridium_complex_from_ligands_canonical(l1, l2, l3, row_index=idx)
            df.at[idx, 'Complex_SMILES_canonical'] = smiles
            df.at[idx, 'Complex_Status_canonical'] = status

            if status == "OK":
                print(f"[{idx}] OK: {smiles}")
            else:
                print(f"[{idx}] {status}")

        except Exception as e:
            print(f"[{idx}] Критическая ошибка: {e}")
            traceback.print_exc()
            df.at[idx, 'Complex_SMILES_canonical'] = None
            df.at[idx, 'Complex_Status_canonical'] = f"Exception: {str(e)}"

    print("Готово.")
    return df


# ============================================================
# ПРИМЕР ЗАПУСКА
# ------------------------------------------------------------
# Ir = pd.read_csv("your_ligands.csv")   # должен содержать L1, L2, L3
Ir = process_dataframe_canonical(Ir)
Ir.to_csv("iridium_complexes_canonical.csv", index=False)
Ir.to_pickle("iridium_complexes_canonical.pkl")
# ============================================================


Обрабатываем 1454 строк для канонических Ir-комплексов...
[0] OK: [Ir+3]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)(n1ccccc1C(=O)[O-])
[1] OK: [Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)(n1ccccc1-c1ccccn1)
[2] OK: [Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)(n1ccc(C(=O)O)cc1-c1cc(C(=O)O)ccn1)
[3] OK: [Ir+3]([O-]S(=O)(=O)c1ccc(P(c2ccc(S(=O)(=O)[O][Na])cc2)c2ccccc2P(c2ccc(S(=O)(=O)[O][Na])cc2)c2ccc(S(=O)(=O)[O][Na])cc2)cc1)([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)ccc1-c1ccccn1)
[4] OK: [Ir+2]([c-]1ccccc1-c1ccccn1)([c-]1ccccc1-c1ccccn1)(n1ccc(P(=O)(OCC)OCC)cc1-c1cc(P(=O)(OCC)OCC)ccn1)
[5] OK: [Ir+2]([c-]1ccccc1-c1nccc2ccccc12)([c-]1ccccc1-c1nccc2ccccc12)(n1ccc(P(=O)(OCC)OCC)cc1-c1cc(P(=O)(OCC)OC

# Opt SMILES

In [18]:
Ir_2 = pd.read_csv('iridium_complexes_canonical.csv')
Ir_2

,L1,L2,L3,Counterion,Abbreviation_in_the_article,Charge,Max_wavelength(nm),PLQY,tau(s*10^-6),Solvent,DOI,Notes,PLQY_in_train,Complex_SMILES,Complex_Status,Complex_SMILES_canonical,Complex_Status_canonical
0,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C([O-])c1ccccn1,NaN,1,0,553,0.2500,4.32,CH2Cl2,10.3390/molecules27010232,NaN,1,O=C1[O-]<-[Ir+3]234567(->[c-]8cc(-c9[n]<-2c2c%...,OK,[Ir+3]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK
1,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,c1ccc(-c2ccccn2)nc1,FP(F)(F)(F)(F)F,2,1,570,0.1700,6.14,CH2Cl2,10.3390/molecules27010232,NaN,1,c1ccc(-[n]23<-[Ir+2]456789(->[c-]%10cc(-c2[n]<...,OK,[Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK
2,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1,FP(F)(F)(F)(F)F,3,1,595,0.1400,0.82,CH3OH,10.3390/molecules27010232,NaN,1,O=C(O)c1cc[n]2<-[Ir+2]345678(->[c-]9cc(-c%10[n...,OK,[Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK
3,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=S(=O)([O-])c1ccc(P(c2ccc(S(=O)(=O)O[Na])cc2)...,NaN,4,0,555,0.0435,41.87,PBS,10.3390/molecules27010232,NaN,1,O=S(=O)([O][Na])c1ccc(P(c2ccc(S(=O)(=O)[O][Na]...,OK,[Ir+3]([O-]S(=O)(=O)c1ccc(P(c2ccc(S(=O)(=O)[O]...,OK
4,[c-]1ccccc1-c1ccccn1,[c-]1ccccc1-c1ccccn1,CCOP(=O)(OCC)c1ccnc(-c2cc(P(=O)(OCC)OCC)ccn2)c1,FP(F)(F)(F)(F)F,Ir(ppy)2bP,1,667,0.0020,NaN,CH3OH,10.1002/adhm.202100706,NaN,1,CCOP(=O)(OCC)c1cc[n]2<-[Ir+2]34(->[c-]5ccccc5-...,OK,[Ir+2]([c-]1ccccc1-c1ccccn1)([c-]1ccccc1-c1ccc...,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,O=C([O-])c1ccccn1,NaN,(m-CzFSOPy)2IrPic,0,554,0.3810,0.64,CH2Cl2,10.1016/j.dyepig.2018.07.019,NaN,1,CCCC[n]12<-[Ir+3]3456(->[O-]C(=O)c7cccc[n]<-37...,OK,[Ir+3]([c-]1cc2c(cc1-c1ccccn1)-c1cc(-c3ccc4c(c...,OK
1450,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir1-me,0,608,0.4000,2.12,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1,Cc1cc2-c3cccc[n]3<-[Ir+3]345(->[c-]6cc(C(F)(F)...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1nccc2ccccc12)([...,OK
1451,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir2-cf3,0,604,0.4100,2.21,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1,FC(F)(F)c1ccc2-c3c4ccccc4cc[n]3<-[Ir+3]345(->[...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1nccc2ccccc12)([...,OK
1452,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PQZ-Ir3-me,0,628,0.6000,2.06,CH2Cl2,10.1016/j.jorganchem.2018.09.009,NaN,1,Cc1cc2-c3cccc[n]3<-[Ir+3]34567(->[c-]8cc(C(F)(...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1ncnc2ccccc12)([...,OK


In [29]:
Ir_2

,L1,L2,L3,Counterion,Abbreviation_in_the_article,Charge,Max_wavelength(nm),PLQY,tau(s*10^-6),Solvent,...,Complex_SMILES,Complex_Status,Complex_SMILES_canonical,Complex_Status_canonical,Complex_XYZ_File,optimized_file,energy,optimization_ok,optimization_error,normalized_smiles
0,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C([O-])c1ccccn1,NaN,1,0,553,0.2500,4.32,CH2Cl2,...,O=C1[O-]<-[Ir+3]234567(->[c-]8cc(-c9[n]<-2c2c%...,OK,[Ir+3]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK,None,optimized_complexes_em_complex/0.xyz,-123.456,True,None,O=C1O<-[Ir+3]234567(->C8cc(-c9[n]<-2c2c%10cccc...
1,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,c1ccc(-c2ccccn2)nc1,FP(F)(F)(F)(F)F,2,1,570,0.1700,6.14,CH2Cl2,...,c1ccc(-[n]23<-[Ir+2]456789(->[c-]%10cc(-c2[n]<...,OK,[Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK,None,optimized_complexes_em_complex/1.xyz,-123.456,True,None,c1ccc(-[n]23<-[Ir+2]456789(->C%10cc(-c2[n]<-4c...
2,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1,FP(F)(F)(F)(F)F,3,1,595,0.1400,0.82,CH3OH,...,O=C(O)c1cc[n]2<-[Ir+2]345678(->[c-]9cc(-c%10[n...,OK,[Ir+2]([c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2cc...,OK,None,optimized_complexes_em_complex/2.xyz,-123.456,True,None,O=C(O)c1cc[n]2<-[Ir+2]345678(->C9cc(-c%10[n]<-...
3,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,[c-]1cc(-c2nc3c4ccccc4c4ccccc4c3n2-c2ccccc2)cc...,O=S(=O)([O-])c1ccc(P(c2ccc(S(=O)(=O)O[Na])cc2)...,NaN,4,0,555,0.0435,41.87,PBS,...,O=S(=O)([O][Na])c1ccc(P(c2ccc(S(=O)(=O)[O][Na]...,OK,[Ir+3]([O-]S(=O)(=O)c1ccc(P(c2ccc(S(=O)(=O)[O]...,OK,None,optimized_complexes_em_complex/3.xyz,-123.456,True,None,O=S(=O)(O[Na])c1ccc(P(c2ccc(S(=O)(=O)O[Na])cc2...
4,[c-]1ccccc1-c1ccccn1,[c-]1ccccc1-c1ccccn1,CCOP(=O)(OCC)c1ccnc(-c2cc(P(=O)(OCC)OCC)ccn2)c1,FP(F)(F)(F)(F)F,Ir(ppy)2bP,1,667,0.0020,NaN,CH3OH,...,CCOP(=O)(OCC)c1cc[n]2<-[Ir+2]34(->[c-]5ccccc5-...,OK,[Ir+2]([c-]1ccccc1-c1ccccn1)([c-]1ccccc1-c1ccc...,OK,None,optimized_complexes_em_complex/4.xyz,-123.456,True,None,CCOP(=O)(OCC)c1cc[n]2<-[Ir+2]34(->C5ccccc5-c5c...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,CCCCn1c2ccccc2c2cc(-c3ccc4c(c3)-c3cc(-c5ccccn5...,O=C([O-])c1ccccn1,NaN,(m-CzFSOPy)2IrPic,0,554,0.3810,0.64,CH2Cl2,...,CCCC[n]12<-[Ir+3]3456(->[O-]C(=O)c7cccc[n]<-37...,OK,[Ir+3]([c-]1cc2c(cc1-c1ccccn1)-c1cc(-c3ccc4c(c...,OK,None,optimized_complexes_em_complex/1449.xyz,-123.456,True,None,CCCC[n]12<-[Ir+3]3456(->OC(=O)c7cccc[n]<-37)(-...
1450,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir1-me,0,608,0.4000,2.12,CH2Cl2,...,Cc1cc2-c3cccc[n]3<-[Ir+3]345(->[c-]6cc(C(F)(F)...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1nccc2ccccc12)([...,OK,None,optimized_complexes_em_complex/1450.xyz,-123.456,True,None,Cc1cc2-c3cccc[n]3<-[Ir+3]345(->C6cc(C(F)(F)F)c...
1451,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2nccc3ccccc23)cc1,FC(F)(F)c1cc(-c2ccccn2)[n-]n1,NaN,PIQ-Ir2-cf3,0,604,0.4100,2.21,CH2Cl2,...,FC(F)(F)c1ccc2-c3c4ccccc4cc[n]3<-[Ir+3]345(->[...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1nccc2ccccc12)([...,OK,None,optimized_complexes_em_complex/1451.xyz,-123.456,True,None,FC(F)(F)c1ccc2-c3c4ccccc4cc[n]3<-[Ir+3]345(->C...
1452,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,FC(F)(F)c1c[c-]c(-c2ncnc3ccccc23)cc1,Cc1cc(-c2ccccn2)[n-]n1,NaN,PQZ-Ir3-me,0,628,0.6000,2.06,CH2Cl2,...,Cc1cc2-c3cccc[n]3<-[Ir+3]34567(->[c-]8cc(C(F)(...,OK,[Ir+3]([c-]1cc(C(F)(F)F)ccc1-c1ncnc2ccccc12)([...,OK,None,optimized_complexes_em_complex/1452.xyz,-123.456,True,None,Cc1cc2-c3cccc[n]3<-[Ir+3]34567(->C8cc(C(F)(F)F...


In [38]:
import re
import pandas as pd
from openbabel import pybel

def normalize_smiles(smiles_str):
    """
    Нормализует SMILES: исправляет радикалы, заряды, изотопы,
    канонизирует структуру через OpenBabel.
    """
    if pd.isna(smiles_str):
        return None
    
    smiles_str = str(smiles_str).strip()
    
    # Пустая строка
    if not smiles_str or smiles_str == '':
        return None
    
    # === ИСПРАВЛЕНИЯ ПЕРЕД ОБРАБОТКОЙ ===
    
    # 1. Удаляем простые соли (они не несут структурной информации)
    # if smiles_str in {'[Cl-]', '[Br-]', '[I-]', '[F-]', '[O-]N=O'}:
    #     return None
    
    # 2. Исправляем явные радикалы и фрагменты
    radical_fixes = {
        '[H]': 'C',      # Водород → метан (минимальная органика)
        '[C]': 'C',      # Радикал → нейтральный углерод
        '[CH]': 'C',     # Метин-радикал → углерод
        '[CH2]': 'C',    # Метилен-радикал → углерод
        '[c-]':'C',
        '[C-]':'C',
        '[O-]':'O',
        '[O]':'O',
        '[CH3]': 'C',    # Метил-радикал → углерод
        '[Si]': '[Si]',  # Кремний оставляем (может быть валиден)
        '[C]1': 'C1',    # Радикал в цикле → нормальный углерод
        '[Cl-]':'[Cl]',
        '[C]#[O]':'C#O',
        '[Br--]':'[Br]',
        '[Br-2]':'[Br]',
        '[CH-]':'C',
        '[n-]':'N',
        '[N-]':'N',
        '[C-2]':'C',
        '[N]':'N',
        '[C@@]':'C@@',
        '[C@]':'C@',
    }
    
    for radical, fixed in radical_fixes.items():
        smiles_str = smiles_str.replace(radical, fixed)
    
    # 3. Исправляем странные формы CO (монооксид углерода)
    smiles_str = smiles_str.replace('[C-2]=O', 'C=O')  # Кетон
    smiles_str = smiles_str.replace('[C#O]', 'C#N')    # Нитрил (ближайший аналог)
    smiles_str = smiles_str.replace('[C]#[O]', 'C#N')
    
    # 4. Убираем заряды (преобразуем в нейтральные формы)
    charge_patterns = [
        (r'\[c-\]', 'c'),
        (r'\[n-\]', 'n'),
        (r'\[s-\]', 's'),
        (r'\[o-\]', 'o'),
        (r'\[O-\]', 'O'),
        (r'\[N-\]', 'N'),
        (r'\[C-\]', 'C'),
        (r'\[S-\]', 'S'),
        (r'\[C\+\]', 'C'),
        (r'\[N\+\]', 'N'),
        (r'\[O\+\]', 'O'),
        (r'\[n\+\]', 'n'),
        (r'\[CH2\]', 'C'),

    ]
    
    for pattern, replacement in charge_patterns:
        smiles_str = re.sub(pattern, replacement, smiles_str)
    
    # 5. Удаляем изотопные метки
    smiles_str = re.sub(r'\[(\d+)([A-Z][a-z]?)\]', r'[\2]', smiles_str)  # [2H] → [H]
    smiles_str = re.sub(r'\[2H\]', '[H]', smiles_str)
    smiles_str = re.sub(r'\[3H\]', '[H]', smiles_str)
    
    # 6. Удаляем только тяжёлые металлы (но оставляем органометаллы с углеродом)
    heavy_metals = ['Zr', 'Ta', 'Sn', 'Ag', 'As', 'Hg', 'Pb', 'Cd', 'Na']
    metal_pattern = r'\[(' + '|'.join(heavy_metals) + r')[^\]]*\]'
    
    # Если SMILES - только металл, удаляем
    if re.match(r'^\[(' + '|'.join(heavy_metals) + r')[^\]]*\]$', smiles_str):
        return None
    
    # Если металл + органика, удаляем только металл
    smiles_str = re.sub(metal_pattern, '', smiles_str)
    
    # 7. Проверка после исправлений
    if not smiles_str or smiles_str.strip() == '':
        return None
    
    # === КАНОНИЗАЦИЯ ЧЕРЕЗ OPENBABEL ===
    try:
        mol = pybel.readstring("smi", smiles_str)
        mol.addh()  # Добавляем водороды
        
        # Канонический SMILES
        canonical_smiles = mol.write("can").strip().split()[0]
        
        return canonical_smiles
        
    except Exception as e:
        # Если OpenBabel не может обработать - возвращаем исправленный вариант
        print(f"Предупреждение: не удалось канонизировать '{smiles_str}': {e}")
        return smiles_str

# === ТЕСТЫ ===
test_cases = list(Ir_2['Complex_SMILES_canonical'])
results = []
from tqdm import tqdm
for smi in tqdm(test_cases):
    result = normalize_smiles(smi)
    results.append(result)
    # print(f"{smi:40s} → {result}")

100%|██████████| 1454/1454 [00:00<00:00, 1992.59it/s]


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
def smiles_to_3d_xtb(smiles, out_xyz='molecule_opt_2.xyz'):
    # Создаем RDKit молекулу из SMILES
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    # Генерируем 3D конформер
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())  
    AllChem.UFFOptimizeMolecule(mol)
    # Сохраняем временный xyz файл
    temp_xyz = 'temp.xyz'
    with open(temp_xyz, 'w') as f:
        f.write(Chem.MolToXYZBlock(mol))
    # Запускаем xtb оптимизацию с выходом в out_xyz
    subprocess.run([
        'xtb', temp_xyz, 
        '--opt', '--xyz', out_xyz
    ], check=True)
    print(f'Оптимизированная структура сохранена в {out_xyz}')
for smi in results:
    


['OC(=O)C1=CC=C[CH]N1[Ir+3](C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1)C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1',
 'c1ccc(cc1)n1c(nc2c1c1ccccc1c1c2cccc1)C1=CC=C(C([CH]1)[Ir+2](N1[CH]C=CC=C1c1ccccn1)C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1)c1ccccn1',
 'OC(=O)C1=C[CH]N(C(=C1)c1nccc(c1)C(=O)O)[Ir+2](C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1)C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1',
 'O=S(=O)(c1ccc(cc1)P(c1ccccc1P(c1ccc(cc1)S(=O)(=O)O)c1ccc(cc1)S(=O)(=O)O)c1ccc(cc1)S(=O)(=O)O)O[Ir+3](C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1)C1[CH]C(=CC=C1c1ccccn1)c1nc2c(n1c1ccccc1)c1ccccc1c1c2cccc1',
 'CCOP(=O)(C1=C[CH]N(C(=C1)c1nccc(c1)P(=O)(OCC)OCC)[Ir+2](C1[CH]C=CC=C1c1ccccn1)C1[CH]C=CC=C1c1ccccn1)OCC',
 'CCOP(=O)(C1=C[CH]N(C(=C1)c1nccc(c1)P(=O)(OCC)OCC)[Ir+2](C1[CH]C=CC=C1c1nccc2c1cccc2)C1[CH]C=CC=C1c1nccc2c1cccc2)OCC',
 'CCOP(=O)(C1=C[CH]N(C(=C1)c1nccc(c1)P(=O)(OCC)OCC)[Ir+2](C1[CH]C=CC=C1c1ccc2c(

In [42]:
pd.DataFrame(results, columns=['SMILES']).to_csv('SMILES_Ir_Em.csv')

# Solvent

In [6]:
sol = df['Solvent']
sol

0       CH2Cl2
1       CH2Cl2
2        CH3OH
3          PBS
4        CH3OH
         ...  
1449    CH2Cl2
1450    CH2Cl2
1451    CH2Cl2
1452    CH2Cl2
1453    CH2Cl2
Name: Solvent, Length: 1454, dtype: object

In [8]:
import pandas as pd
import numpy as np

# Словарь названий растворителей и их SMILES
solvent_to_smiles = {
    '2-MeTHF': 'CC1COCCO1',              # 2-Methyltetrahydrofuran
    'C2H5OH': 'CCO',                     # Ethanol
    'CH2Cl2': 'ClCCl',                   # Dichloromethane
    'CH3CN': 'CC#N',                     # Acetonitrile
    'CH3OH': 'CO',                       # Methanol
    'CHCl3': 'ClC(Cl)Cl',                # Chloroform
    'DCE': 'ClCCCl',                     # 1,2-Dichloroethane
    'DMF': 'CN(C)C=O',                   # Dimethylformamide
    'DMSO': 'CS(=O)C',                   # Dimethyl sulfoxide
    'H2O': 'O',                          # Water
    'PBS': None,                         # Phosphate Buffered Saline — не молекула, нет SMILES
    'THF': 'C1CCOC1',                    # Tetrahydrofuran
    'acetone': 'CC(=O)C',                # Acetone
    'cyclohexane': 'C1CCCCC1',           # Cyclohexane
    'toluene': 'Cc1ccccc1',              # Toluene
    np.nan: None                         # Обработка NaN
}

# Исходный список растворителей (в том порядке, как у вас)
solvents = [
    '2-MeTHF',
    'C2H5OH',
    'CH2Cl2',
    'CH3CN',
    'CH3OH',
    'CHCl3',
    'DCE',
    'DMF',
    'DMSO',
    'H2O',
    'PBS',
    'THF',
    'acetone',
    'cyclohexane',
    np.nan,
    'toluene'
]

# Создаём DataFrame
df = pd.DataFrame({'Solvent': solvents})

# Добавляем столбец SMILES с использованием map
df['SMILES_Solvent'] = df['Solvent'].map(solvent_to_smiles)

# Выводим результат
print(df)

        Solvent SMILES_Solvent
0       2-MeTHF      CC1COCCO1
1        C2H5OH            CCO
2        CH2Cl2          ClCCl
3         CH3CN           CC#N
4         CH3OH             CO
5         CHCl3      ClC(Cl)Cl
6           DCE         ClCCCl
7           DMF       CN(C)C=O
8          DMSO        CS(=O)C
9           H2O              O
10          PBS           None
11          THF        C1CCOC1
12      acetone        CC(=O)C
13  cyclohexane       C1CCCCC1
14          NaN           None
15      toluene      Cc1ccccc1


In [10]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import os

# Название папки для сохранения
OUTPUT_DIR = "optimized_solvents_Ir"

# Создаём папку, если её нет
os.makedirs(OUTPUT_DIR, exist_ok=True)

def smiles_to_3d_xtb(smiles, out_xyz_path):
    """
    Преобразует SMILES в 3D-структуру, оптимизирует её с помощью xTB и сохраняет в указанный путь.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("RDKit не смог распарсить SMILES")
        
        mol = Chem.AddHs(mol)
        res = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        if res == -1:
            raise RuntimeError("Не удалось сгенерировать 3D-конформер")
        
        AllChem.UFFOptimizeMolecule(mol)
        
        temp_xyz = 'temp_mol.xyz'
        with open(temp_xyz, 'w') as f:
            f.write(Chem.MolToXYZBlock(mol))
        
        result = subprocess.run([
            'xtb', temp_xyz, '--opt', '--xyz', out_xyz_path
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            raise RuntimeError(f"xTB failed:\n{result.stderr}")
        
        if os.path.exists(temp_xyz):
            os.remove(temp_xyz)
            
        print(f"✅ Успешно: {out_xyz_path}")
        return True
        
    except Exception as e:
        print(f"❌ Ошибка для SMILES '{smiles}': {e}")
        return False

def safe_filename(name):
    """Преобразует название в безопасное имя файла."""
    name = str(name).strip()
    return "".join(c if c.isalnum() or c in ('-', '_') else '_' for c in name)

# --- Данные ---
solvent_to_smiles = {
    '2-MeTHF': 'CC1COCCO1',
    'C2H5OH': 'CCO',
    'CH2Cl2': 'ClCCl',
    'CH3CN': 'CC#N',
    'CH3OH': 'CO',
    'CHCl3': 'ClC(Cl)Cl',
    'DCE': 'ClCCCl',
    'DMF': 'CN(C)C=O',
    'DMSO': 'CS(=O)C',
    'H2O': 'O',
    'PBS': None,
    'THF': 'C1CCOC1',
    'acetone': 'CC(=O)C',
    'cyclohexane': 'C1CCCCC1',
    'toluene': 'Cc1ccccc1'
}

df = pd.DataFrame({
    'Solvent': list(solvent_to_smiles.keys()),
    'SMILES': list(solvent_to_smiles.values())
})

# Убираем строки без SMILES
df = df.dropna(subset=['SMILES'])
df = df[df['SMILES'] != '']

# Обработка каждой молекулы
for _, row in df.iterrows():
    solvent = row['Solvent']
    smiles = row['SMILES']
    
    safe_name = safe_filename(solvent)
    out_path = os.path.join(OUTPUT_DIR, f"{safe_name}.xyz")
    
    print(f"\nОбрабатываю: {solvent} → {out_path}")
    smiles_to_3d_xtb(smiles, out_xyz_path=out_path)


Обрабатываю: 2-MeTHF → optimized_solvents_Ir/2-MeTHF.xyz
✅ Успешно: optimized_solvents_Ir/2-MeTHF.xyz

Обрабатываю: C2H5OH → optimized_solvents_Ir/C2H5OH.xyz
✅ Успешно: optimized_solvents_Ir/C2H5OH.xyz

Обрабатываю: CH2Cl2 → optimized_solvents_Ir/CH2Cl2.xyz
✅ Успешно: optimized_solvents_Ir/CH2Cl2.xyz

Обрабатываю: CH3CN → optimized_solvents_Ir/CH3CN.xyz
✅ Успешно: optimized_solvents_Ir/CH3CN.xyz

Обрабатываю: CH3OH → optimized_solvents_Ir/CH3OH.xyz
✅ Успешно: optimized_solvents_Ir/CH3OH.xyz

Обрабатываю: CHCl3 → optimized_solvents_Ir/CHCl3.xyz
✅ Успешно: optimized_solvents_Ir/CHCl3.xyz

Обрабатываю: DCE → optimized_solvents_Ir/DCE.xyz
✅ Успешно: optimized_solvents_Ir/DCE.xyz

Обрабатываю: DMF → optimized_solvents_Ir/DMF.xyz
✅ Успешно: optimized_solvents_Ir/DMF.xyz

Обрабатываю: DMSO → optimized_solvents_Ir/DMSO.xyz
✅ Успешно: optimized_solvents_Ir/DMSO.xyz

Обрабатываю: H2O → optimized_solvents_Ir/H2O.xyz
✅ Успешно: optimized_solvents_Ir/H2O.xyz

Обрабатываю: THF → optimized_solvents_

In [44]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# Словарь растворителей
solvent_to_smiles = {
    '2-MeTHF': 'CC1COCCO1',
    'C2H5OH': 'CCO',
    'CH2Cl2': 'ClCCl',
    'CH3CN': 'CC#N',
    'CH3OH': 'CO',
    'CHCl3': 'ClC(Cl)Cl',
    'DCE': 'ClCCCl',
    'DMF': 'CN(C)C=O',
    'DMSO': 'CS(=O)C',
    'H2O': 'O',
    'PBS': None,
    'THF': 'C1CCOC1',
    'acetone': 'CC(=O)C',
    'cyclohexane': 'C1CCCCC1',
    'toluene': 'Cc1ccccc1'
}

# Создаём папку для сохранения xyz-файлов
os.makedirs("optimized_solvents_xyz", exist_ok=True)

def mol_to_xyz(mol, filename):
    """Сохраняет RDKit-молекулу в формат .xyz"""
    conf = mol.GetConformer()
    atoms = mol.GetAtoms()
    n_atoms = mol.GetNumAtoms()

    with open(filename, 'w') as f:
        f.write(f"{n_atoms}\n")
        f.write("Optimized with RDKit\n")
        for atom, pos in zip(atoms, conf.GetPositions()):
            symbol = atom.GetSymbol()
            x, y, z = pos
            f.write(f"{symbol:2s} {x:12.6f} {y:12.6f} {z:12.6f}\n")

# Обработка каждой молекулы
for name, smiles in solvent_to_smiles.items():
    if smiles is None:
        print(f"Skipping {name} (no SMILES provided)")
        continue

    print(f"Processing {name} ({smiles})...")

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"❌ Failed to parse SMILES for {name}")
        continue

    mol = Chem.AddHs(mol)

    # Embed 3D coordinates
    success = AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    if success != 0:
        print(f"❌ Embedding failed for {name}")
        continue

    # Optimize: MMFF first
    ff_result = AllChem.MMFFOptimizeMolecule(mol)
    if ff_result == -1:
        print(f"⚠️ MMFF not applicable for {name}, trying UFF...")
        uff_result = AllChem.UFFOptimizeMolecule(mol)
        if uff_result == -1:
            print(f"❌ Both MMFF and UFF failed for {name}")
            continue

    # Сохраняем в .xyz
    out_path = os.path.join("optimized_solvents_xyz", f"{name}.xyz")
    mol_to_xyz(mol, out_path)
    print(f"✅ Saved to {out_path}")

print("\n✨ All done! XYZ files are in 'optimized_solvents_xyz/'")

Processing 2-MeTHF (CC1COCCO1)...
✅ Saved to optimized_solvents_xyz/2-MeTHF.xyz
Processing C2H5OH (CCO)...
✅ Saved to optimized_solvents_xyz/C2H5OH.xyz
Processing CH2Cl2 (ClCCl)...
✅ Saved to optimized_solvents_xyz/CH2Cl2.xyz
Processing CH3CN (CC#N)...
✅ Saved to optimized_solvents_xyz/CH3CN.xyz
Processing CH3OH (CO)...
✅ Saved to optimized_solvents_xyz/CH3OH.xyz
Processing CHCl3 (ClC(Cl)Cl)...
✅ Saved to optimized_solvents_xyz/CHCl3.xyz
Processing DCE (ClCCCl)...
✅ Saved to optimized_solvents_xyz/DCE.xyz
Processing DMF (CN(C)C=O)...
✅ Saved to optimized_solvents_xyz/DMF.xyz
Processing DMSO (CS(=O)C)...
✅ Saved to optimized_solvents_xyz/DMSO.xyz
Processing H2O (O)...
✅ Saved to optimized_solvents_xyz/H2O.xyz
Skipping PBS (no SMILES provided)
Processing THF (C1CCOC1)...
✅ Saved to optimized_solvents_xyz/THF.xyz
Processing acetone (CC(=O)C)...
✅ Saved to optimized_solvents_xyz/acetone.xyz
Processing cyclohexane (C1CCCCC1)...
✅ Saved to optimized_solvents_xyz/cyclohexane.xyz
Processing to

In [9]:
df.to_csv('Sol_Ir.csv')